# Group 16 · V4 Pre-S4 Baseline on Tesla T4

## 1. Experiment Overview

Notebook thu bằng chứng còn thiếu cho P10-T01 trước khi đánh giá V5. Đối chứng là V4 production, với ba cặp `site1/large`, `site2/large`, `site3/large`. Ca chính hiện chốt tạm `site2/large` theo quy tắc dự phòng trong [contract](docs/report/p10-v5-contract.md).

Kết quả gồm timing trước S4, timing A/B, thời gian host chờ count, mẫu VRAM, panorama đối chiếu, biểu đồ và ZIP evidence. Probe dùng source overlay từ chính checkout, thêm event và một lần chờ ở ranh giới S4. Các kernel và thứ tự stage V4 được giữ. Lượt production được chạy riêng để nhận biết overhead của probe.

Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**. Khi một cell báo lỗi, xem log và chạy cell Export để tải phần evidence đã có.


## 2. Colab Environment Verification

Notebook ghi model GPU thực, driver, compute capability, toolkit, CMake và NSYS. T4 được yêu cầu mặc định. Thay REQUIRE_T4 chỉ khi chủ động thu một lượt chẩn đoán trên GPU khác.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/nh-atuan/panorama-stitching.git"
SOURCE_REF = "refs/heads/chuong/p10-v5"  # @param {type:"string"}
DATASET_INPUT = ""  # @param {type:"string"}
REQUIRE_T4 = True  # @param {type:"boolean"}
RUN_NSYS = True  # @param {type:"boolean"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
DRIVE_EXPORT_DIR = ""  # @param {type:"string"}
RUNS = 5  # @param {type:"integer"}

if RUNS < 5:
    raise ValueError("At least five measured runs are required.")
WORK = Path(tempfile.mkdtemp(prefix="group16_p10_t01_"))
EVIDENCE = WORK / "Group16_P10_T01_PreS4_Evidence"
EVIDENCE.mkdir()
(EVIDENCE / "logs").mkdir()
SUMMARY = None
ENVIRONMENT = {"time_utc": datetime.now(timezone.utc).isoformat(), "source_ref": SOURCE_REF}

def command_text(args, required=True):
    result = subprocess.run(args, capture_output=True, text=True)
    if required and result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout.strip() if result.returncode == 0 else "unavailable"

if not shutil.which("nvidia-smi") or not shutil.which("nvcc"):
    raise RuntimeError("CUDA GPU and nvcc are required. Select Colab T4 GPU and rerun.")
gpu = command_text(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free,compute_cap",
                    "--format=csv,noheader,nounits"])
ENVIRONMENT.update(gpu_query=gpu, nvcc=command_text(["nvcc", "--version"]),
                   cmake=command_text(["cmake", "--version"], required=False) if shutil.which("cmake") else "unavailable",
                   nsys=command_text(["nsys", "--version"], required=False) if shutil.which("nsys") else "unavailable")
if REQUIRE_T4 and not all("T4" in line.split(",")[0] for line in gpu.splitlines()):
    raise RuntimeError(f"Requested Tesla T4, actual GPU: {gpu}")
print(json.dumps(ENVIRONMENT, indent=2))
(EVIDENCE / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")


## 3. Repository Setup

SOURCE_REF chọn nhánh hoặc commit chứa notebook và instrumentation. Fetch đúng ref đã nhập và ghi SHA thực. Khi chạy lại toàn bộ notebook, một workspace evidence mới được tạo.


In [ ]:
PROJECT = WORK / "panorama-stitching"
subprocess.run(["git", "init", str(PROJECT)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", REPOSITORY, SOURCE_REF], cwd=PROJECT, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=PROJECT, check=True)
SOURCE_SHA = command_text(["git", "-C", str(PROJECT), "rev-parse", "HEAD"])
required = ("scripts/build_pres4_probe.py", "scripts/pres4_baseline.py",
            "tests/tools/p10_pres4_probe.cuh", "src/v4/v4_pipeline.cu",
            "docs/report/p10-v5-contract.md")
for name in required:
    if not (PROJECT / name).is_file():
        raise FileNotFoundError(f"{SOURCE_SHA} lacks required file {name}")
sys.path.insert(0, str(PROJECT / "scripts"))
import pres4_baseline as baseline
from run_benchmark import parse_version_csv
ENVIRONMENT["source_sha"] = SOURCE_SHA
baseline.dump(EVIDENCE / "environment.json", ENVIRONMENT)
(EVIDENCE / "source_sha.txt").write_text(SOURCE_SHA + "\n", encoding="utf-8")
shutil.copy2(PROJECT / "docs/report/p10-v5-contract.md", EVIDENCE / "contract_at_run.md")
print("Measured source SHA:", SOURCE_SHA)


## 4. Dataset Preparation

Dùng dataset cuối kỳ từ Drive đã sử dụng ở notebook P9. DATASET_INPUT có thể trỏ đến thư mục chứa manifest hoặc file ZIP đã có. Khi để trống, tải ZIP chính thức. Kiểm hash đủ dataset trước build. Ảnh large phải có cùng kích thước với manifest đã nghiệm thu.


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

from midterm_notebook import safe_extract
if DATASET_INPUT:
    dataset_input = Path(DATASET_INPUT)
else:
    from fetch_and_prepare_dataset import ensure_gdown
    ensure_gdown()
    import gdown
    download = WORK / "dataset_download"
    gdown.download_folder(
        url="https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL?usp=sharing",
        output=str(download) + "/", use_cookies=False)
    archives = list(download.rglob("dataset.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Expected official dataset.zip. Set DATASET_INPUT to an existing copy.")
    dataset_input = archives[0]
if dataset_input.is_file():
    extracted = WORK / "dataset"
    safe_extract(dataset_input, extracted)
    candidates = list(extracted.rglob("manifest.csv"))
else:
    candidates = list(dataset_input.rglob("manifest.csv"))
if len(candidates) != 1:
    raise ValueError("Dataset input must contain exactly one manifest.csv.")
DATASET = candidates[0].parent
INPUT_MANIFEST = baseline.dataset_manifest(DATASET)
if len(INPUT_MANIFEST) != 6:
    raise ValueError("Missing large input image.")
baseline.dump(EVIDENCE / "input_manifest.json", INPUT_MANIFEST)
shutil.copy2(DATASET / "manifest.csv", EVIDENCE / "dataset_manifest.csv")
for row in INPUT_MANIFEST:
    print(row["file"], row["width"], row["height"], (DATASET / row["file"]).is_file(), row["sha256"])


## 5. Build V4 Production

Repo dùng `scripts/build.py` với `sm_75 -O2 --fmad=false`. Build production trước, rồi build probe từ overlay có anchor kiểm tra nghiêm ngặt. Source gốc trong `src/` không bị chỉnh sửa. Giữ các file overlay và lệnh build trong evidence để review.


In [ ]:
record = lambda args, label, **kw: baseline.recorded(args, EVIDENCE, label, PROJECT, **kw)
build = [sys.executable, "scripts/build.py", "--compiler", "nvcc", "--out-dir", "build/production"]
if REQUIRE_T4:
    build.append("--require-t4")
record(build, "build_production")
record([sys.executable, "scripts/build_pres4_probe.py", "--out-dir", "build/pres4"], "build_probe")
PRODUCTION = PROJECT / "build/production/stitch"
PROBE = PROJECT / "build/pres4/stitch"
ENVIRONMENT.update(production_sha256=baseline.sha256(PRODUCTION), probe_sha256=baseline.sha256(PROBE))
baseline.dump(EVIDENCE / "environment.json", ENVIRONMENT)
shutil.copytree(PROJECT / "build/pres4/overlay", EVIDENCE / "instrumented_source")
shutil.copy2(PROJECT / "build/pres4/build_commands.json", EVIDENCE / "probe_build_commands.json")


## 6. Identify and Instrument the Pre-S4 Boundary

Probe ghi origin event ngay trước khi submit H2D(A). Ready event được ghi trên cùng stream sau S3(B), trước launch S4. CUDA elapsed origin → ready đo trực tiếp đoạn trước S4, gồm khoảng trống do host chậm phát lệnh.

Probe chờ ready trên host để đo `Host_PreS4_ms` đến khi feature hai ảnh thực sự sẵn sàng. Vì có thêm event và readiness wait, `Total_ms` của probe chứa overhead instrumentation. Lượt production là đối chứng timing riêng. Không dùng tổng S1–S3 thay cho đoạn origin → ready.


## 7. Stage and Branch Timing

V4 giữ thứ tự H2D(A), H2D(B), S1(A/B), S2(A/B), S3(A/B). Event bao quanh từng nhánh đo interval trên stream. `CountD2H_A/B_ms` bao quanh submission copy count và có thể chứa overhead host, nhất là do pageable candidate storage của V4. Đây là event span, chưa xác định duration copy thuần từ hardware trace.

`CountWait_A/B_ms` là thời gian CPU nằm trong cudaStreamSynchronize của S2. Nó gồm GPU work đang chờ hoàn tất, không phải chi phí API thuần. Probe giữ cùng phép tính và dữ liệu candidate như production.


## 8. Repeated Baseline Measurements

Mỗi binary tự chạy một warm-up và ít nhất năm lượt đo cho mỗi ca. Chạy production rồi probe, không chạy đồng thời. Chạy lại cell này cần workspace mới để tránh nối trùng CSV. Giữ toàn bộ run và xuất mean, median, min, max cùng sample standard deviation.


In [ ]:
for site in baseline.SITES:
    probe_csv = EVIDENCE / f"{site}_probe.csv"
    if probe_csv.exists():
        raise RuntimeError("Measurements already exist. Start a fresh notebook run.")
    left, right = [DATASET / site / f"large_{side}.jpg" for side in ("left", "right")]
    for kind, executable in (("production", PRODUCTION), ("probe", PROBE)):
        env = dict(os.environ)
        if kind == "probe":
            env["P10_PROBE_CSV"] = str(probe_csv)
            env["P10_PROBE_ENV"] = str(EVIDENCE / f"{site}_gpu.json")
        record([executable, left, right, EVIDENCE / f"{site}_{kind}.png",
                "--version", "4", "--runs", str(RUNS), "--debug"],
               f"{site}_{kind}", env=env)
    print(site, "measurement completed")


## 9. GPU Memory Measurements

Probe ghi free/total trước workspace, ở đầu mỗi lượt và sau D2H bằng cudaMemGetInfo. Mẫu đầu lượt đo thường còn canvas của warm-up hoặc lượt trước. `Sampled_max_device_used_bytes` là VRAM dùng lớn nhất tại các checkpoint trên toàn device. Peak allocation chính xác của process chưa có nếu chưa phân tích allocation timeline.

Nhu cầu scratch một nhánh được tính bằng chính các scratch calculators S1–S3 trên dimensions/Config hiện hành. So headroom với tổng scratch A+B thêm vào baseline là phép kiểm tra bảo thủ, giữ cả scratch downstream. Đây là ước lượng trước khi V5 có allocator thực. Các resource metadata/event và cấp phát phát sinh của V5 còn cần được tính khi triển khai.


In [ ]:
for site in baseline.SITES:
    info = json.loads((EVIDENCE / f"{site}_gpu.json").read_text())
    rows = baseline.read_probe(EVIDENCE / f"{site}_probe.csv", RUNS)
    print(site, json.dumps(info, indent=2))
    print("Workspace free checkpoints:", [r["WorkspaceFree_bytes"] for r in rows])
    print("After D2H free checkpoints:", [r["AfterD2HFree_bytes"] for r in rows])
    if info["compute_major"] != 7 or info["compute_minor"] != 5:
        print("Actual compute capability differs from the sm_75 target.")


## 10. NSYS Profiling (Optional)

NSYS chạy production riêng với một warm-up và một lượt đo. Trace chứa H2D, kernels, API synchronization và NVTX S1–S4. SQLite được xuất để trích GPU idle và các event theo correlation ID sau này. Notebook giữ nguyên trace và trạng thái export. Khi không có NSYS hoặc profiler lỗi, timing/memory vẫn được giữ và trạng thái profiler ghi rõ.

Xem `S2.score_nms`, cudaStreamSynchronize và thời điểm đầu kernel trong `S4.match`. Tổng duration có thể khác wall time khi operations chồng lấn. Chỉ gán số GPU idle sau khi xác định run boundary và hợp các interval kernel/copy trong cùng cửa sổ.


In [ ]:
PROFILE_STATUS = []
for site in baseline.SITES:
    base = EVIDENCE / "profiling" / site
    base.mkdir(parents=True, exist_ok=True)
    args = ["nsys", "profile", "--trace=cuda,nvtx,osrt", "--cuda-memory-usage=true",
            "--force-overwrite=true", "-o", base / "capture",
            PRODUCTION, DATASET / site / "large_left.jpg", DATASET / site / "large_right.jpg",
            base / "panorama.png", "--version", "4", "--runs", "1", "--debug"]
    status = {"case": site + "/large", "command": list(map(str, args)), "available": False}
    if RUN_NSYS and shutil.which("nsys"):
        code, _, _ = record(args, f"{site}_nsys", required=False)
        trace = base / "capture.nsys-rep"
        status.update(returncode=code, available=(code == 0 and trace.is_file()))
        if status["available"]:
            code, _, _ = record(["nsys", "export", "--type", "sqlite", "--force-overwrite=true",
                                  "-o", base / "capture.sqlite", trace],
                                 f"{site}_sqlite", required=False)
            status["sqlite_export_ok"] = code == 0 and (base / "capture.sqlite").is_file()
    else:
        status["reason"] = "NSYS disabled or unavailable"
    PROFILE_STATUS.append(status)
baseline.dump(EVIDENCE / "profiling_status.json", PROFILE_STATUS)
print(json.dumps(PROFILE_STATUS, indent=2))


## 11. Results Summary

Bảng và biểu đồ lấy từ file thực nghiệm. Panorama probe phải trùng hash PNG production trước khi tổng hợp. File `*_statistics.json` chứa đủ sáu thống kê cho từng metric. Bảng báo song song Total production và Total probe, với pre-S4 share tính trong chính lượt probe.


In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage
SUMMARY = baseline.analyze(EVIDENCE, RUNS)
columns = list(SUMMARY[0])
table = "| " + " | ".join(columns) + " |\n"
table += "|" + "---|" * len(columns) + "\n"
for row in SUMMARY:
    table += "| " + " | ".join(f"{row[k]:.4f}" if isinstance(row[k], float) else str(row[k])
                                for k in columns) + " |\n"
display(Markdown(table))
baseline.plot_summary(SUMMARY, EVIDENCE / "pres4_comparison.png")
display(DisplayImage(filename=str(EVIDENCE / "pres4_comparison.png")))
print("Raw CSV, full statistics JSON, summary and memory tables saved.")


## 12. Primary Benchmark Case Assessment

Giữ lựa chọn đã chốt `site2/large` trong contract. Bảng mới chỉ ra ca có pre-S4 share lớn nhất để đề nghị rà lại lựa chọn trước khi xem kết quả V5. Đánh giá thêm độ lệch chuẩn, thời gian host chờ S2 và headroom từng ca.

Mức share lớn hơn tạo dư địa theo tỷ lệ; khả năng overlap thực vẫn phụ thuộc contention và lịch phát việc. Kết quả gợi ý không tự thay ca chính. Ghi quyết định cuối cùng vào contract trước đo V5.


In [ ]:
ASSESSMENT = baseline.assessment(SUMMARY)
ASSESSMENT["source_sha"] = SOURCE_SHA
ASSESSMENT["gpu_query"] = ENVIRONMENT["gpu_query"]
baseline.dump(EVIDENCE / "primary_case_assessment.json", ASSESSMENT)
print(json.dumps(ASSESSMENT, indent=2, ensure_ascii=False))


## 13. Export Experimental Evidence

Chạy cell này cả khi bước đo bị lỗi để tải log và trạng thái hiện có. ZIP có manifest checksum. Có thể chép ZIP sang thư mục Drive đã mount bằng DRIVE_EXPORT_DIR. Evidence chưa có summary được đánh dấu partial. Mỗi file xuất tương ứng với artifact đã thực sự tạo.


In [ ]:
if "baseline" not in globals():
    raise RuntimeError("Repository helpers have not loaded. Fix source setup first.")
baseline.dump(EVIDENCE / "experiment_status.json", {
    "timing_complete": SUMMARY is not None,
    "t4_model_checked": REQUIRE_T4,
    "source_sha": SOURCE_SHA,
    "profiling": globals().get("PROFILE_STATUS", []),
    "limitations": [
        "GPU idle and exact process allocation peak require trace review.",
        "Production and probe timing are separate experimental records.",
    ],
})
archive = baseline.package(EVIDENCE)
print("Evidence ZIP:", archive)
print("SHA-256:", baseline.sha256(archive))
if DRIVE_EXPORT_DIR:
    destination = Path(DRIVE_EXPORT_DIR)
    if not destination.is_dir():
        raise FileNotFoundError("Drive export directory must already exist and be mounted.")
    shutil.copy2(archive, destination / archive.name)
from google.colab import files
files.download(str(archive))


## 14. Reproduction and Next Steps

Mở notebook từ nhánh `chuong/p10-v5`, chọn T4 và chạy từ trên xuống. Giữ dataset cuối kỳ theo manifest P7–P9. Source ref có thể đổi thành commit SHA đã publish, mọi lượt sẽ lưu checkout SHA thực.

Nếu build hoặc timing lỗi, đọc `logs/*.stderr` và chạy cell Export. Chỉ nhận một lượt hoàn chỉnh khi đủ ba ca, mỗi ca đủ số run, stdout CLI hợp lệ và panorama probe khớp production. Review `summary.csv`, `raw_timing.csv`, `memory.csv`, statistics JSON, source SHA và profiler status trong ZIP.

Chương dùng ZIP để bổ sung bảng pre-S4, thời gian A/B, host wait và VRAM vào [contract T01](docs/report/p10-v5-contract.md). Đánh giá lại lựa chọn ca chính và ghi quyết định trước khi quan sát performance V5. Các khoảng GPU idle và allocation peak cần trích từ trace nếu có. Sau đó Tuấn dùng contract cho nghiệm thu V5 và Đế đo V5 chính thức sau khi đã đạt correctness.
